In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install dagshub mlflow -q

import mlflow
import dagshub

dagshub.init(repo_owner='lbegi23', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 638.1 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 2.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 25.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 29.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=144777e8-adf1-4570-8fb5-3fb5217def82&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=4b6acbb07b6b26b3f40d926405eac5ec345d47fdb990afbcd33a05a4bcb17c64




Accessing as lbegi23

Initialized MLflow to track repo "lbegi23/IEEE-CIS-Fraud-Detection"

Repository lbegi23/IEEE-CIS-Fraud-Detection initialized!

In [3]:
model_uri = "models:/xgboost-tuned-pipeline/2"
pipeline = mlflow.sklearn.load_model(model_uri)

print("Pipeline loaded successfully")
print(pipeline)

Pipeline loaded successfully
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['V40', 'V45', 'C7', 'V295',
                                                   'V91', 'V70', 'C1', 'C14',
                                                   'V15', 'V62', 'C4', 'V298',
                                                   'V286', 'V312', 'V48',
                                                   'V315', 'C13', 'V281', 'V87',
                                                   'C2', 'D3', 'card3', 'V133',
                                                   'C11', 'V53', 'V76', 'V296',
                                                   'V303', 'V317', 'C8', ...]),
                                                 ('cat',
           

In [4]:
transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

df_test = transaction.merge(identity, on='TransactionID', how='left')

print(df_test.shape)

(506691, 433)


In [5]:
high_null_cols = df_test.columns[df_test.isnull().mean() > 0.5].tolist()
high_cardinality_cols = ['DeviceInfo', 'id_33', 'id_31', 'id_30']
cols_to_drop = list(set(high_null_cols + high_cardinality_cols))

df_test_clean = df_test.drop(columns=[c for c in cols_to_drop if c in df_test.columns])

df_test_clean['TransactionAmt_log']   = np.log1p(df_test_clean['TransactionAmt'])
df_test_clean['TransactionAmt_cents'] = df_test_clean['TransactionAmt'] % 1
df_test_clean['hour'] = (df_test_clean['TransactionDT'] // 3600) % 24
df_test_clean['day']  = (df_test_clean['TransactionDT'] // (3600 * 24)) % 7

pipeline_features = pipeline.named_steps['preprocessor'].transformers_[0][2] + \
                    pipeline.named_steps['preprocessor'].transformers_[1][2]

df_test_final = df_test_clean[pipeline_features]

print(f"Test shape after cleaning: {df_test_final.shape}")

Test shape after cleaning: (506691, 165)


In [6]:
preds = pipeline.predict_proba(df_test_final)[:, 1]

submission = pd.DataFrame({
    'TransactionID': transaction['TransactionID'],
    'isFraud': preds
})

submission.to_csv('submission.csv', index=False)

print(f"Submission shape: {submission.shape}")
print(submission.head())

Submission shape: (506691, 2)
   TransactionID   isFraud
0        3663549  0.003054
1        3663550  0.002650
2        3663551  0.003133
3        3663552  0.004049
4        3663553  0.003058
